# Experiment 3 ? Differential Learning Rate

**Hypothesis:**

The ImageNet-pretrained backbone already contains useful visual features and should be updated conservatively, while the newly initialized classifier needs larger updates. Using a lower learning rate for the backbone than for the classifier may preserve transferable features while adapting the prediction head faster.

**Controlled change:**

Use two AdamW parameter groups: `1e-5` for the backbone and `1e-4` for `model.model.fc`.

The 320?320 data pipeline, pretrained ResNet-50, cross-entropy objective, weight decay, AMP, seed, validation-only checkpoint selection, and no-test-evaluation policy follow Experiment 1. There is no scheduler.

In [ ]:
import gc, json, os, sys, time
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists(): PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src').exists(): raise RuntimeError('Could not find src')
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from datasets import load_dataset
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
    classification_report, cohen_kappa_score, confusion_matrix,
    precision_recall_fscore_support)
from torch.utils.data import DataLoader
from torchvision import transforms
from src.datasets.glaucoma_dataset import GlaucomaDataset, IDX_TO_CLASS
from src.models.resnet50 import ResNet50Classifier
from src.training.config import (CHECKPOINT_DIR, NUM_CLASSES, RESULTS_FIGURES_DIR,
    RESULTS_METRICS_DIR, SEED, WEIGHT_DECAY, set_seed)

if not torch.cuda.is_available(): raise RuntimeError('CUDA is not available')
DEVICE = torch.device('cuda:0')
set_seed(SEED)
sns.set_theme(style='whitegrid')
print('Project root:', PROJECT_ROOT)
print('PyTorch:', torch.__version__, '| GPU:', torch.cuda.get_device_name(0))

## Configuration

Experiment 3 starts independently from torchvision's ImageNet-pretrained ResNet-50 weights. No Experiment 1 or 2 checkpoint is loaded. The maximum epoch budget is 40 with patience 7; checkpoint selection still uses the lowest validation cross-entropy loss.

In [ ]:
EXPERIMENT_NAME = 'EXPERIMENT_3_DIFFERENTIAL_LR'
BASELINE_NAME = 'EXPERIMENT_1_320'
IMAGE_SIZE, BATCH_SIZE = 320, 16
BATCH_SIZE_CANDIDATES = (16, 8)
AMP_ENABLED = True
BACKBONE_LR, CLASSIFIER_LR = 1e-5, 1e-4
EPOCHS, PATIENCE = 40, 7
CLASS_NAMES = [IDX_TO_CLASS[i] for i in range(NUM_CLASSES)]

CHECKPOINT_PATH = Path(CHECKPOINT_DIR) / 'exp3_differential_lr_best.pt'
METRICS_PATH = Path(RESULTS_METRICS_DIR) / 'exp3_differential_lr_metrics.json'
HISTORY_PATH = Path(RESULTS_METRICS_DIR) / 'exp3_differential_lr_history.json'
FIGURE_DIR = Path(RESULTS_FIGURES_DIR)
CONFUSION_MATRIX_PATH = FIGURE_DIR / 'exp3_differential_lr_confusion_matrix.png'
CURVES_PATH = FIGURE_DIR / 'exp3_differential_lr_training_curves.png'
LR_CURVE_PATH = FIGURE_DIR / 'exp3_differential_lr_learning_rates.png'

assert IMAGE_SIZE == 320 and WEIGHT_DECAY == 1e-4
assert BACKBONE_LR == 1e-5 and CLASSIFIER_LR == 1e-4
assert EPOCHS == 40 and PATIENCE == 7
assert 'exp3_differential_lr' in CHECKPOINT_PATH.name
print({'seed': SEED, 'image_size': IMAGE_SIZE, 'requested_batch_size': BATCH_SIZE,
       'amp': AMP_ENABLED, 'backbone_lr': BACKBONE_LR, 'classifier_lr': CLASSIFIER_LR,
       'optimizer': 'AdamW', 'weight_decay': WEIGHT_DECAY,
       'epochs': EPOCHS, 'patience': PATIENCE,
       'checkpoint_criterion': 'lowest validation CrossEntropyLoss',
       'test_evaluated': False})

## Unchanged Experiment 1 data pipeline

The transforms and official train/validation split are identical to Experiment 1. The test split is not loaded or evaluated.

In [ ]:
train_transform_320 = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.05),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
eval_transform_320 = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
raw_dataset = load_dataset('moondream/glaucoma-detection')
assert [len(raw_dataset[name]) for name in ('train', 'validation', 'test')] == [2847, 1259, 1272]

def build_loaders(batch_size):
    train = GlaucomaDataset(raw_dataset['train'], transform=train_transform_320)
    val = GlaucomaDataset(raw_dataset['validation'], transform=eval_transform_320)
    return (DataLoader(train, batch_size=batch_size, shuffle=True),
            DataLoader(val, batch_size=batch_size, shuffle=False))

def verify_loaders(train_loader, val_loader):
    assert [len(x.dataset) for x in (train_loader, val_loader)] == [2847, 1259]
    images, labels = next(iter(train_loader))
    print('Image batch:', tuple(images.shape))
    print('Label batch:', tuple(labels.shape))
    assert images.shape[1:] == (3, 320, 320)
    assert labels.min() >= 0 and labels.max() < NUM_CLASSES

train_loader, val_loader = build_loaders(BATCH_SIZE)
verify_loaders(train_loader, val_loader)

## ImageNet-pretrained ResNet-50 with differential learning rates

The classifier parameters (`model.model.fc`) use `1e-4`; every other trainable parameter uses `1e-5`. Both groups use AdamW with the shared `1e-4` weight decay and no scheduler.

In [ ]:
def build_training_objects():
    model = ResNet50Classifier(
        num_classes=NUM_CLASSES, pretrained=True, freeze_features=False).to(DEVICE)
    backbone_parameters = [
        parameter for name, parameter in model.model.named_parameters()
        if not name.startswith('fc.')]
    classifier_parameters = list(model.model.fc.parameters())
    optimizer = torch.optim.AdamW([
        {'params': backbone_parameters, 'lr': BACKBONE_LR, 'name': 'backbone'},
        {'params': classifier_parameters, 'lr': CLASSIFIER_LR, 'name': 'classifier'},
    ], weight_decay=WEIGHT_DECAY)

    backbone_ids = {id(parameter) for parameter in backbone_parameters}
    classifier_ids = {id(parameter) for parameter in classifier_parameters}
    trainable_ids = {id(parameter) for parameter in model.parameters() if parameter.requires_grad}
    assert not backbone_ids & classifier_ids
    assert backbone_ids | classifier_ids == trainable_ids
    assert [group['lr'] for group in optimizer.param_groups] == [BACKBONE_LR, CLASSIFIER_LR]
    assert model.model.fc.in_features == 2048 and model.model.fc.out_features == NUM_CLASSES
    return model, optimizer

loss_fn = nn.CrossEntropyLoss()
probe_model, probe_optimizer = build_training_objects()
print('Optimizer groups:', [(group['name'], group['lr'], len(group['params']))
                             for group in probe_optimizer.param_groups])
print(f'Trainable parameters: {sum(p.numel() for p in probe_model.parameters() if p.requires_grad):,}')
del probe_model, probe_optimizer
torch.cuda.empty_cache()

## Training

Training uses cross-entropy and AMP. Early stopping and checkpoint selection use validation loss only. The learning rate of each parameter group is recorded every epoch.

In [ ]:
def compute_metrics(labels, predictions):
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, labels=range(NUM_CLASSES), average='macro', zero_division=0)
    return {'accuracy': float(accuracy_score(labels, predictions)),
            'balanced_accuracy': float(balanced_accuracy_score(labels, predictions)),
            'macro_precision': float(precision), 'macro_recall': float(recall),
            'macro_f1': float(f1),
            'qwk': float(cohen_kappa_score(labels, predictions, weights='quadratic'))}

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss, predictions, labels_all = 0.0, [], []
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        outputs = model(images)
        total_loss += loss_fn(outputs, labels).item() * images.size(0)
        predictions.extend(outputs.argmax(1).cpu().tolist())
        labels_all.extend(labels.cpu().tolist())
    metrics = compute_metrics(labels_all, predictions)
    metrics['loss'] = float(total_loss / len(loader.dataset))
    return metrics, predictions, labels_all

def fit(model, optimizer, train_loader, val_loader):
    history = {'train_loss': [], 'val_loss': [], 'val_metrics': [],
               'learning_rates': {'backbone': [], 'classifier': []}}
    scaler = torch.amp.GradScaler('cuda', enabled=AMP_ENABLED)
    best_loss, best_epoch, best_metrics, stale = float('inf'), None, None, 0
    for epoch in range(1, EPOCHS + 1):
        group_lrs = {group['name']: group['lr'] for group in optimizer.param_groups}
        model.train()
        total_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            with torch.autocast('cuda', dtype=torch.float16, enabled=AMP_ENABLED):
                loss = loss_fn(model(images), labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            total_loss += loss.item() * images.size(0)
        train_loss = total_loss / len(train_loader.dataset)
        metrics, _, _ = evaluate(model, val_loader)
        history['train_loss'].append(float(train_loss))
        history['val_loss'].append(metrics['loss'])
        history['val_metrics'].append(metrics)
        for group_name, learning_rate in group_lrs.items():
            history['learning_rates'][group_name].append(learning_rate)
        improved = metrics['loss'] < best_loss
        print(f"Epoch {epoch:02d}/{EPOCHS} | Train Loss {train_loss:.4f} | "
              f"Validation Loss {metrics['loss']:.4f} | Validation Accuracy {metrics['accuracy']:.4f} | "
              f"Validation Balanced Accuracy {metrics['balanced_accuracy']:.4f} | "
              f"Validation Macro Precision {metrics['macro_precision']:.4f} | "
              f"Validation Macro Recall {metrics['macro_recall']:.4f} | "
              f"Validation Macro F1 {metrics['macro_f1']:.4f} | Validation QWK {metrics['qwk']:.4f} | "
              f"Backbone LR {group_lrs['backbone']:.1e} | Classifier LR {group_lrs['classifier']:.1e}"
              + (' (best validation loss)' if improved else ''))
        if improved:
            best_loss, best_epoch, best_metrics, stale = metrics['loss'], epoch, metrics.copy(), 0
            CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
            torch.save(model.state_dict(), CHECKPOINT_PATH)
        else:
            stale += 1
        if stale >= PATIENCE:
            print(f'No validation-loss improvement for {PATIENCE} epochs; stopping.')
            break
    return history, best_epoch, best_metrics

oom_occurred = False
for batch_size in BATCH_SIZE_CANDIDATES:
    gc.collect(); torch.cuda.empty_cache(); set_seed(SEED)
    train_loader, val_loader = build_loaders(batch_size)
    print(f'\nAttempting Experiment 3 with batch size {batch_size}')
    verify_loaders(train_loader, val_loader)
    set_seed(SEED)
    model, optimizer = build_training_objects()
    attempt_started = time.perf_counter()
    try:
        history, best_epoch, best_val_metrics = fit(model, optimizer, train_loader, val_loader)
        training_time_seconds = time.perf_counter() - attempt_started
        actual_batch_size = batch_size
        break
    except RuntimeError as error:
        if 'out of memory' not in str(error).lower(): raise
        oom_occurred = True
        print(f'CUDA OOM at batch size {batch_size}; restarting from ImageNet initialization at epoch 1.')
        del model, optimizer
else:
    raise RuntimeError('CUDA OOM at batch sizes 16 and 8')

assert CHECKPOINT_PATH.exists()
assert all(lr == BACKBONE_LR for lr in history['learning_rates']['backbone'])
assert all(lr == CLASSIFIER_LR for lr in history['learning_rates']['classifier'])
print({'epochs_completed': len(history['train_loss']), 'best_epoch': best_epoch,
       'best_validation_loss': best_val_metrics['loss'], 'actual_batch_size': actual_batch_size,
       'backbone_lr': BACKBONE_LR, 'classifier_lr': CLASSIFIER_LR,
       'training_time_seconds': training_time_seconds, 'amp': AMP_ENABLED})

## Experiment 3 validation histories

In [ ]:
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
epochs_ran = list(range(1, len(history['train_loss']) + 1))
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
axes[0, 0].plot(epochs_ran, history['train_loss'], marker='o', label='Train')
axes[0, 0].plot(epochs_ran, history['val_loss'], marker='o', label='Validation')
axes[0, 0].set(title='Loss', xlabel='Epoch'); axes[0, 0].legend()
for key, label in [('accuracy', 'Accuracy'), ('balanced_accuracy', 'Balanced accuracy')]:
    axes[0, 1].plot(epochs_ran, [m[key] for m in history['val_metrics']], marker='o', label=label)
axes[0, 1].set(title='Validation accuracy', xlabel='Epoch'); axes[0, 1].legend()
for key, label in [('macro_precision', 'Macro precision'), ('macro_recall', 'Macro recall'),
                   ('macro_f1', 'Macro F1')]:
    axes[1, 0].plot(epochs_ran, [m[key] for m in history['val_metrics']], marker='o', label=label)
axes[1, 0].set(title='Validation macro metrics', xlabel='Epoch'); axes[1, 0].legend()
axes[1, 1].plot(epochs_ran, [m['qwk'] for m in history['val_metrics']], marker='o')
axes[1, 1].set(title='Validation QWK', xlabel='Epoch')
fig.suptitle('Experiment 3: ResNet-50 320x320 differential learning rate')
fig.tight_layout(); fig.savefig(CURVES_PATH, dpi=160); plt.show()

plt.figure(figsize=(8, 5))
plt.plot(epochs_ran, history['learning_rates']['backbone'], marker='o', label='Backbone')
plt.plot(epochs_ran, history['learning_rates']['classifier'], marker='o', label='Classifier')
plt.xlabel('Epoch'); plt.ylabel('Learning rate'); plt.yscale('log')
plt.title('Experiment 3 parameter-group learning rates'); plt.legend()
plt.tight_layout(); plt.savefig(LR_CURVE_PATH, dpi=160); plt.show()

## Selected-checkpoint validation evaluation

This reloads only the checkpoint produced by Experiment 3, then reports validation metrics, per-class results, all six directed severity-error counts, and the confusion matrix.

In [ ]:
model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=True))
selected_val_metrics, val_predictions, val_labels = evaluate(model, val_loader)
assert abs(selected_val_metrics['loss'] - best_val_metrics['loss']) < 1e-8
report_dict = classification_report(
    val_labels, val_predictions, labels=range(NUM_CLASSES), target_names=CLASS_NAMES,
    digits=4, zero_division=0, output_dict=True)
print(json.dumps(selected_val_metrics, indent=2))
print(classification_report(val_labels, val_predictions, labels=range(NUM_CLASSES),
      target_names=CLASS_NAMES, digits=4, zero_division=0))

matrix = confusion_matrix(val_labels, val_predictions, labels=range(NUM_CLASSES))
ordinal_errors = {
    'Normal -> Early': int(matrix[0, 1]), 'Normal -> Advanced': int(matrix[0, 2]),
    'Early -> Normal': int(matrix[1, 0]), 'Early -> Advanced': int(matrix[1, 2]),
    'Advanced -> Early': int(matrix[2, 1]), 'Advanced -> Normal': int(matrix[2, 0]),
}
print('Directed ordinal errors:', json.dumps(ordinal_errors, indent=2))

plt.figure(figsize=(7, 6))
sns.heatmap(matrix, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel('Predicted'); plt.ylabel('Ground truth')
plt.title('Experiment 3 validation confusion matrix'); plt.tight_layout()
plt.savefig(CONFUSION_MATRIX_PATH, dpi=160); plt.show()

## Save Experiment 3 artifacts and compare with Experiment 1

The comparison is validation-only. Experiment 1 aggregate metrics are read from its saved metrics file when available; unsupported class-level comparisons are not inferred.

In [ ]:
METRICS_PATH.parent.mkdir(parents=True, exist_ok=True)
history_record = {
    'experiment': EXPERIMENT_NAME, 'seed': SEED, 'image_size': IMAGE_SIZE,
    'batch_size': actual_batch_size, 'backbone_learning_rate': BACKBONE_LR,
    'classifier_learning_rate': CLASSIFIER_LR, 'history': history,
}
with HISTORY_PATH.open('w', encoding='utf-8') as file:
    json.dump(history_record, file, indent=2)

exp1_path = Path(RESULTS_METRICS_DIR) / 'resnet50_320.json'
exp1_results, exp1_metrics = {}, {}
if exp1_path.exists():
    with exp1_path.open(encoding='utf-8') as file: exp1_results = json.load(file)
    exp1_metrics = exp1_results.get('validation_metrics', {})

rows = [('Validation loss', 'loss'), ('Accuracy', 'accuracy'),
        ('Balanced Accuracy', 'balanced_accuracy'), ('Macro F1', 'macro_f1'), ('QWK', 'qwk')]
print('\nMetric'.ljust(26), BASELINE_NAME.rjust(20), EXPERIMENT_NAME.rjust(32))
print('-' * 80)
for label, key in rows:
    old = f"{exp1_metrics[key]:.4f}" if key in exp1_metrics else 'not available'
    print(label.ljust(26), old.rjust(20), f"{selected_val_metrics[key]:.4f}".rjust(32))

class_recall = {name: float(report_dict[name]['recall']) for name in CLASS_NAMES}
print('\nExperiment 3 class-level recall:', json.dumps(class_recall, indent=2))
if not exp1_results.get('classification_report'):
    print('Experiment 1 class-level recall/confusion matrix/history are not present in its metrics file; '
          'no unsupported class-level or dynamics comparison is made.')
print('Inspect validation loss, all metrics, class recall, confusion patterns, and curves before drawing a conclusion.')

results = {
    'experiment': EXPERIMENT_NAME, 'baseline': BASELINE_NAME,
    'description': 'Experiment 3 — Differential Learning Rate',
    'model': 'ResNet50Classifier(pretrained=True, freeze_features=False)',
    'initialization': 'torchvision ImageNet pretrained weights; newly initialized model.model.fc',
    'class_mapping': {str(i): CLASS_NAMES[i] for i in range(NUM_CLASSES)},
    'seed': SEED, 'image_size': IMAGE_SIZE, 'batch_size': actual_batch_size,
    'amp_enabled': AMP_ENABLED, 'cuda_oom_occurred': oom_occurred,
    'epochs_completed': len(history['train_loss']), 'maximum_epochs': EPOCHS,
    'early_stopping_patience': PATIENCE, 'best_epoch': best_epoch,
    'best_validation_loss': selected_val_metrics['loss'],
    'checkpoint_criterion': 'lowest validation CrossEntropyLoss',
    'optimizer': 'AdamW', 'backbone_learning_rate': BACKBONE_LR,
    'classifier_learning_rate': CLASSIFIER_LR, 'weight_decay': WEIGHT_DECAY,
    'scheduler': None, 'training_time_seconds': training_time_seconds,
    'validation_metrics': selected_val_metrics,
    'classification_report': report_dict, 'confusion_matrix': matrix.tolist(),
    'directed_ordinal_errors': ordinal_errors, 'test_evaluated': False,
    'artifacts': {'checkpoint': str(CHECKPOINT_PATH), 'history': str(HISTORY_PATH),
                  'confusion_matrix': str(CONFUSION_MATRIX_PATH),
                  'training_curves': str(CURVES_PATH), 'learning_rates': str(LR_CURVE_PATH)},
}
with METRICS_PATH.open('w', encoding='utf-8') as file:
    json.dump(results, file, indent=2)
print('Saved Experiment 3 artifacts:', CHECKPOINT_PATH, METRICS_PATH, HISTORY_PATH,
      CONFUSION_MATRIX_PATH, CURVES_PATH, LR_CURVE_PATH, sep='\n')